# Camera Calibration

This notebook combines the calibration and testing steps into a single workflow.

## 1. Camera Calibration

This step reads the calibration images, finds the chessboard corners, and calculates the camera matrix and distortion coefficients. The results are saved to `basler_calib_data.npz`.

In [4]:
import numpy as np
import glob
import cv2
import os

In [5]:
CHECKERBOARD = (9, 7)
SQUARE_SIZE = 0.01 #in meters
IMAGE_DIR = 'calib_images/*.jpg'
if SQUARE_SIZE == 0.000:
    raise ValueError("ERROR: You forgot to update the SQUARE_SIZE variable at the top!")

In [ ]:
criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 30, 0.001)#maximum of 30 iterations and maximum of 0.001 pixels

objp = np.zeros((CHECKERBOARD[0] * CHECKERBOARD[1], 3), np.float32)#empty array of checkerboard size
objp[:, :2] = np.mgrid[0:CHECKERBOARD[0], 0:CHECKERBOARD[1]].T.reshape(-1, 2)#fills the array with coordinates
objp = objp * SQUARE_SIZE #now it represents the size of the checkerboard

objpoints = []
imgpoints = []

images = glob.glob(IMAGE_DIR)
img_shape = None

print(f"Found {len(images)} images. Starting processing...\n")

successful_images = 0

Found 44 images. Starting heavy processing...

[1/44] Scanning calib_000.jpg... Corners found!
[2/44] Scanning calib_001.jpg... Corners found!
[3/44] Scanning calib_002.jpg... Corners found!
[4/44] Scanning calib_003.jpg... Corners found!
[5/44] Scanning calib_004.jpg... Corners found!
[6/44] Scanning calib_005.jpg... Corners found!
[7/44] Scanning calib_006.jpg... Corners found!
[8/44] Scanning calib_007.jpg... Corners found!
[9/44] Scanning calib_008.jpg... Corners found!
[10/44] Scanning calib_009.jpg... Corners found!
[11/44] Scanning calib_010.jpg... Corners found!
[12/44] Scanning calib_011.jpg... Corners found!
[13/44] Scanning calib_012.jpg... Corners found!
[14/44] Scanning calib_013.jpg... Corners found!
[15/44] Scanning calib_014.jpg... Corners found!
[16/44] Scanning calib_015.jpg... Corners found!
[17/44] Scanning calib_016.jpg... Corners found!
[18/44] Scanning calib_017.jpg... Corners found!
[19/44] Scanning calib_018.jpg... Corners found!
[20/44] Scanning calib_019.jpg.

In [ ]:
for i, fname in enumerate(images):
    filename = os.path.basename(fname)
    print(f"[{i+1}/{len(images)}] Scanning {filename}...", end=" ", flush=True)
    
    img = cv2.imread(fname)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    
    if img_shape is None:
        img_shape = gray.shape[::-1]
    
    # --- THE HIGH-RESOLUTION FIX ---
    # These flags clean up lighting and noise before looking for the corners
    flags = cv2.CALIB_CB_ADAPTIVE_THRESH + cv2.CALIB_CB_NORMALIZE_IMAGE + cv2.CALIB_CB_FAST_CHECK
    ret, corners = cv2.findChessboardCorners(gray, CHECKERBOARD, flags)
    
    if ret:
        objpoints.append(objp)
        # Refine corner locations to sub-pixel accuracy
        corners_refined = cv2.cornerSubPix(gray, corners, (11,11), (-1,-1), criteria)
        imgpoints.append(corners_refined)
        successful_images += 1
        print("Corners found!")
    else:
        print("FAILED to find all corners.")

print(f"\nSuccessfully extracted corners from {successful_images} out of {len(images)} images.")

if successful_images == 0:
    print("\nCRITICAL ERROR: Could not find the board in ANY image.")
    print("Fix 1: Try changing CHECKERBOARD to (10, 8) instead of (8, 10).")
    print("Fix 2: Ensure your images aren't too blurry or poorly lit.")
    raise RuntimeError("Calibration failed. No corners found.")

print("Calculating final camera matrix and distortion... (Almost done!)")

# Perform the actual calibration
ret, mtx, dist, rvecs, tvecs = cv2.calibrateCamera(objpoints, imgpoints, img_shape, None, None)

print("\n" + "="*30)
print("--- CALIBRATION RESULTS ---")
print("="*30)
print(f"Reprojection Error: {ret:.4f} pixels")
print("\nCamera Matrix (Intrinsics):\n", mtx)
print("\nDistortion Coefficients:\n", dist.ravel())

# Save the data for your live application
np.savez('basler_calib_data.npz', mtx=mtx, dist=dist)
print("\nSuccess! Data saved to 'basler_calib_data.npz'")

## 2. Check Result (Live Undistortion)

Connects to the camera and uses the saved calibration data to show the live undistorted image.

In [7]:
from pypylon import pylon
import cv2
import numpy as np

# Load your new calibration data
with np.load('basler_calib_data.npz') as data:
    mtx = data['mtx']
    dist = data['dist']

DISPLAY_SCALE = 0.4 # Will now comfortably fit your screen since it's only one image

# Connect to the camera
camera = pylon.InstantCamera(pylon.TlFactory.GetInstance().CreateFirstDevice())
camera.StartGrabbing(pylon.GrabStrategy_LatestImageOnly)
converter = pylon.ImageFormatConverter()
converter.OutputPixelFormat = pylon.PixelType_BGR8packed

print("Live Undistortion Started!")
print("-> Press SPACEBAR to flip between RAW and CALIBRATED.")
print("-> Press 'Q' to quit.")

show_calibrated = True # Start by showing the good, flat image

try:
    while camera.IsGrabbing():
        grabResult = camera.RetrieveResult(5000, pylon.TimeoutHandling_ThrowException)
        
        if grabResult.GrabSucceeded():
            image = converter.Convert(grabResult)
            img = image.GetArray()
            
            # Determine which image to show based on the spacebar toggle
            if show_calibrated:
                # Apply the mathematical correction
                display_img = cv2.undistort(img, mtx, dist, None, mtx)
                label = "CALIBRATED / FLAT (Press SPACE to toggle)"
                color = (0, 255, 0) # Green text
            else:
                # Show the raw image
                display_img = img.copy()
                label = "RAW / DISTORTED (Press SPACE to toggle)"
                color = (0, 0, 255) # Red text
                
            # Resize to fit your screen
            h, w = display_img.shape[:2]
            new_w, new_h = int(w * DISPLAY_SCALE), int(h * DISPLAY_SCALE)
            display_img_small = cv2.resize(display_img, (new_w, new_h))
            
            # Draw the label so you know what you are looking at
            cv2.putText(display_img_small, label, (20, 40), 
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, color, 2)
            
            cv2.imshow('Calibration Proof', display_img_small)
            
            key = cv2.waitKey(1) & 0xFF
            
            # Press 'Q' to quit
            if key == ord('q'):
                break
            # Press SPACEBAR to flip the image
            elif key == ord(' '): 
                show_calibrated = not show_calibrated

        grabResult.Release()
finally:
    camera.StopGrabbing()
    cv2.destroyAllWindows()


Live Undistortion Started!
-> Press SPACEBAR to flip between RAW and CALIBRATED.
-> Press 'Q' to quit.
